This code is for preparing data relatd to CVD, AD, and covariates for hierarchical clustering analysis. Analysis is run on AoU participants with EHR data.



In [ ]:
#read in necessary libraries
library(tidyverse)



##Load in AD/CVD data for correct cohort


In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "AD_CVD_diagnoses" for domain "observation" and was generated for All of Us Controlled Tier Dataset v7
dataset_24103539_observation_sql <- paste("
    SELECT
        observation.person_id,
        o_source_concept.concept_name as source_concept_name,
        o_source_concept.concept_code as source_concept_code,
        o_source_concept.vocabulary_id as source_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `observation` observation 
        WHERE
            (
                observation_source_concept_id IN (44830102, 45552870)
            )  
            AND (
                observation.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) observation 
    LEFT JOIN
        `concept` o_source_concept 
            ON observation.observation_source_concept_id = o_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
observation_24103539_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "observation_24103539",
  "observation_24103539_*.csv")
message(str_glue('The data will be written to {observation_24103539_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_24103539_observation_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  observation_24103539_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {observation_24103539_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_24103539_observation_df <- read_bq_export_from_workspace_bucket(observation_24103539_path)

dim(dataset_24103539_observation_df)

head(dataset_24103539_observation_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "AD_CVD_diagnoses" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_24103539_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (312327, 313217, 315286, 316139, 320128, 321318, 378419, 443454)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (1569114, 1569115, 1569116, 1569133, 1569134, 1569135, 1569145, 1569162, 1569419, 1569420, 35207643, 35207644, 35207645, 35207646, 35207647, 35207648, 35207649, 35207650, 35207651, 35207652, 35207653, 35207654, 35207655, 35207656, 35207657, 35207658, 35207659, 35207660, 35207661, 35207662, 35207663, 35207664, 35207665, 35207666, 35207667, 35207702, 35207703, 35207704, 35207705, 35207706, 35207766, 35207767, 35207768, 35207769, 35207770, 35207771, 35207915, 35207916, 35207917, 35207918, 44819738, 44821971, 44823135, 44826658, 44832409, 44833593, 44833594, 45533439, 45538368, 45538373, 45538374, 45538375, 45538377, 45543167, 45543168, 45548010, 45548011, 45548012, 45548013, 45552792, 45552871, 45557538, 45557539, 45562343, 45562344, 45562351, 45567167, 45567168, 45576866, 45576868, 45581766, 45581775, 45586572, 45586573, 45586574, 45586575, 45591456, 45591458, 45591459, 45591460, 45591461, 45591537, 45596197, 45596198, 45596199, 45596204, 45601024,
 45601026, 45601027, 45601028, 45605777, 45605784, 45605785, 45605786, 45605787, 45605788)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_24103539_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_24103539",
  "condition_24103539_*.csv")
message(str_glue('The data will be written to {condition_24103539_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_24103539_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_24103539_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_24103539_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_24103539_condition_df <- read_bq_export_from_workspace_bucket(condition_24103539_path)

dim(dataset_24103539_condition_df)

head(dataset_24103539_condition_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "AD_CVD_diagnoses" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_24103539_person_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth 
    FROM
        `person` person   
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_24103539_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_24103539",
  "person_24103539_*.csv")
message(str_glue('The data will be written to {person_24103539_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_24103539_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_24103539_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_24103539_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_24103539_person_df <- read_bq_export_from_workspace_bucket(person_24103539_path)

dim(dataset_24103539_person_df)

head(dataset_24103539_person_df, 5)


## Extract Diagnoses


In [ ]:
ad_cvd <- rbind(dataset_24103539_condition_df, dataset_24103539_observation_df) %>% select(source_concept_code, everything())

#initialize list
icd_codes <- list(

#save all icd10 codes in a list
Code = list( a = 'I10',
             b = 'I95',
             c = 'I20',
             d = 'I21',
             e = 'I26',
             f = 'I48',
             g = 'I50',
             h = 'I44',
             i = c('I05', 'I06', 'I07', 'I08', 'I09'),
             j = 'I25',
             k = 'I63',
             l = 'G30'
),

#Save disease names in second element of list
Disease = c('Hypertension', 'Hypotension', 'Angina.Pectoris', 'Acute.Myocardial.Infarction', 'Pulmonary.Embolism', 'Atrial.Fibriliation', 'Heart.Failure', 'Blockage', 'Chronic.Rheumatic.Heart.Disease', 'Chronic.Ischemic.Heart.Disease', 'Cerebral.Infarction', 'Alzheimers.Disease')

)

#function to search for ICD 10 diagnosis of AD or CVD
icd_lookup <- function(icd10_codes, data, file_name, output_dir = "C:/Users/agtoy/Desktop/URIP/UKB Cleaned Data") {
  # Ensure necessary packages are loaded
  library(tidyverse)
  library(stringr)
  
  # Check that `icd10_codes` has the correct structure
  if (!is.list(icd10_codes) || length(icd10_codes) != 2) {
    stop("icd10_codes must be a list with two elements: numerical codes and disease names.")
  }
  
  # Ensure the data has at least one column
  if (ncol(data) < 1) {
    stop("The data frame must have at least one column.")
  }
  
  # Iterate through the ICD-10 codes
  for (i in seq_along(icd10_codes[[1]])) {
    # Check if current code exists in the data
    code_matches <- sapply(icd10_codes[[1]][[i]], function(p) grepl(p, data[[1]]))
    data[[icd10_codes[[2]][[i]]]] <- apply(code_matches, 1, any)
  }
  
  # Save the updated data frame to the specified directory
  if (!dir.exists(output_dir)) {
    stop("The specified output directory does not exist.")
  }
  write.csv(data, file.path(output_dir, paste0(file_name, ".csv")), row.names = FALSE)
}

icd_lookup(icd_codes, ad_cvd, 'ad_cvd_data', "~/")

cvd <- read.csv("~/ad_cvd_data.csv")
cvd <- cvd %>% 
  select(-source_concept_code, -source_vocabulary, -source_concept_name) %>% group_by(person_id) %>% 
  summarise(across(everything(), sum)) %>% 
    mutate(across(-person_id, ~ .x != 0))

#join with the demographics dataframe
cvd <- cvd %>% 
  right_join(dataset_24103539_person_df) %>% 
  mutate(across(everything(), ~ replace(.x, is.na(.x), FALSE)))



##Load other covariates from "clustering" dataset


In [ ]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "Clustering" for domain "person" and was generated for All of Us Controlled Tier Dataset v7
dataset_20265581_person_sql <- paste("
    SELECT
        person.person_id,
        person.birth_datetime as date_of_birth,
        p_race_concept.concept_name as race,
        p_ethnicity_concept.concept_name as ethnicity,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_20265581_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_20265581",
  "person_20265581_*.csv")
message(str_glue('The data will be written to {person_20265581_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_20265581_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_20265581_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_20265581_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(race = col_character(), ethnicity = col_character(), sex_at_birth = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_20265581_person_df <- read_bq_export_from_workspace_bucket(person_20265581_path)

dim(dataset_20265581_person_df)

head(dataset_20265581_person_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "Clustering" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_20265581_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (440383)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (1326492, 1326493, 1567956, 1567958, 1567959, 1567960, 1567964, 1567965, 1567966, 1567969, 1567971, 35206881, 35206882, 37200198, 37200199, 37200200, 37200201, 37200202, 37200203, 37200204, 37200205, 37200206, 37200207, 37200208, 37200209, 37200210, 37200211, 37200212, 37200213, 37200214, 37200215, 37200216, 37200217, 37200218, 37200219, 37200220, 37200221, 37200222, 37200223, 37200224, 37200225, 37200227, 37200228, 37200229, 37200230, 37200232, 37200233, 37200234, 37200235, 37200237, 37200238, 37200239, 37200240, 37200242, 37200243, 37200244, 37200245, 37200246, 37200247, 37200248, 37200249, 37200251, 37200252, 37200253, 37200254, 45533019, 45533020, 45533021, 45533022, 45533023, 45537961, 45537962, 45542738, 45547625, 45547626, 45547627, 45552385, 45552386, 45557112, 45557113, 45561949, 45566731, 45576443, 45581352, 45581353, 45581354, 45581355, 45586139, 45586140, 45591027, 45591029, 45591030, 45591031, 45595797, 45595798, 45595799, 45600641,
 45600642, 45605401, 45605402, 45605403, 45605404, 45605405)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_20265581_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_20265581",
  "condition_20265581_*.csv")
message(str_glue('The data will be written to {condition_20265581_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_20265581_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_20265581_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_20265581_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_20265581_condition_df <- read_bq_export_from_workspace_bucket(condition_20265581_path)

dim(dataset_20265581_condition_df)

head(dataset_20265581_condition_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "Clustering" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v7
dataset_20265581_measurement_sql <- paste("
    SELECT
        measurement.person_id,
        m_standard_concept.concept_name as standard_concept_name,
        measurement.measurement_datetime,
        measurement.value_as_number,
        m_unit.concept_name as unit_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (3004249, 3007070, 3012888, 3028288, 3038553)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) measurement 
    LEFT JOIN
        `concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id 
    LEFT JOIN
        `concept` m_unit 
            ON measurement.unit_concept_id = m_unit.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurement_20265581_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurement_20265581",
  "measurement_20265581_*.csv")
message(str_glue('The data will be written to {measurement_20265581_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_20265581_measurement_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  measurement_20265581_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurement_20265581_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), unit_concept_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_20265581_measurement_df <- read_bq_export_from_workspace_bucket(measurement_20265581_path)

dim(dataset_20265581_measurement_df)

head(dataset_20265581_measurement_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "Clustering" for domain "drug" and was generated for All of Us Controlled Tier Dataset v7
dataset_20265581_drug_sql <- paste("
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `cb_criteria` cr             
                        WHERE
                            concept_id IN (1307046, 1307863, 1308842, 1310149, 1313200, 1314002, 1317640, 1318137, 1319880, 1319998, 1322081, 1326012, 1327978, 1328165, 1332418, 1338005, 1345858, 1346686, 1346823, 1347384, 1351557, 1353766, 1353776, 1367500, 1367571, 1386957, 1395058, 1510813, 1539403, 1545958, 1549686, 1551860, 40226742, 902427, 904542, 907013, 932745, 948787, 956874, 978555, 991382, 992590)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) d_exposure 
    LEFT JOIN
        `concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_20265581_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "drug_20265581",
  "drug_20265581_*.csv")
message(str_glue('The data will be written to {drug_20265581_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_20265581_drug_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  drug_20265581_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {drug_20265581_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_20265581_drug_df <- read_bq_export_from_workspace_bucket(drug_20265581_path)

dim(dataset_20265581_drug_df)

head(dataset_20265581_drug_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "Clustering" for domain "survey" and was generated for All of Us Controlled Tier Dataset v7
dataset_20265581_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.question,
        answer.answer  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (1585375, 1585857, 1585940, 1586140, 1586201, 836807)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_ehr_data = 1 ) )
        )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_20265581_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_20265581",
  "survey_20265581_*.csv")
message(str_glue('The data will be written to {survey_20265581_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_20265581_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_20265581_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_20265581_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_20265581_survey_df <- read_bq_export_from_workspace_bucket(survey_20265581_path)

dim(dataset_20265581_survey_df)

head(dataset_20265581_survey_df, 5)



##Data cleaning on covariates


In [ ]:
#need to replace the number of this df, but other than that this should be pretty straightforward
condition_covars <- dataset_20265581_condition_df

head(condition_covars)

#extract correct labeling for depression and type 2 diabetes diagnoses
condition_covars <- condition_covars %>%
  rename(diagnosis = standard_concept_name) %>% 
  mutate(diagnosis = if_else(str_detect(diagnosis, regex("Depress", ignore_case = TRUE))|str_detect(diagnosis, regex("dysthymia", ignore_case = TRUE))|str_detect(diagnosis, regex("seasonal", ignore_case = TRUE))|str_detect(diagnosis, regex("dysphor", ignore_case = TRUE))|str_detect(diagnosis, regex("blues", ignore_case = TRUE)),
                             "Depression",
                             diagnosis))%>% 
  mutate(diagnosis = if_else(str_detect(diagnosis, regex("type 2", ignore_case = TRUE)),
                             "Diabetes",
                             diagnosis)) %>% 
  filter(diagnosis %in% c("Depression", "Diabetes")) %>% 
    pivot_wider(
      id_cols = person_id,
    names_from = diagnosis,
    values_from = diagnosis,
    values_fn = length,  # Use length to count occurrences of each diagnosis
    values_fill = list(diagnosis = 0)  # Fill missing values with 0 (FALSE)
  ) %>% 
    mutate(across(starts_with("Depression"):starts_with("Diabetes"), ~ . != 0))



##Clean measurement covariates


In [ ]:
#this will need to be expanded to work for hdl, ldl, sbp, dbp
meas_covars <- dataset_20265581_measurement_df
head(meas_covars)
library(dplyr)
library(tidyr)

meas_names <- unique(meas_covars$standard_concept_name)
#this code was developed by chat, and hopefully will be useful for averaging out over all measurements
average_measurements <- function(df, variables, id_col = "person_id", type_col = "standard_concept_name", value_col = "value_as_number") {
  df %>%
    filter(.data[[type_col]] %in% variables) %>%
    group_by(.data[[id_col]], .data[[type_col]]) %>%
    summarise(mean_value = mean(.data[[value_col]], na.rm = TRUE), .groups = "drop") %>%
    pivot_wider(names_from = .data[[type_col]], values_from = mean_value)
}

#find average measurements across all readings
avg_meas_covars <- average_measurements(meas_covars, meas_names)

#remove HDL/LDL columns due to too high missingness %
avg_meas_covars <- avg_meas_covars %>% 
  select(-`Cholesterol in HDL [Mass/volume] in Serum or Plasma`, -`Cholesterol in LDL [Mass/volume] in Serum or Plasma by calculation`)

#replace names for better access
colnames(avg_meas_covars) <- c("person_id", "BMI", "DBP", "SBP")



##Clean survey covariates


In [ ]:
surv_covars <- dataset_20265581_survey_df

#pivot wider for question columns, answer values
surv_covars <- surv_covars %>% 
  pivot_wider(names_from = question,
              values_from = answer,
              values_fn = toString) 

#recode survey response values
surv_covars <- surv_covars%>% 
  #recode categorical variables, combining non-answer responses
  mutate(
    Income = ifelse(`Income: Annual Income` %in% c("PMI: Prefer Not To Answer", "PMI: Skip"), NA, `Income: Annual Income`),
    Income = factor(Income),
    Race = gsub("What Race Ethnicity: ", "", `Race: What Race Ethnicity`, fixed = TRUE),
    Race = ifelse(Race %in% c('PMI: Prefer Not To Answer', 'PMI: Skip'), NA,
                ifelse(grepl(',', Race), 'Mixed', Race)),
    Race = factor(Race),
    Education = ifelse(`Education Level: Highest Grade` %in% c("PMI: Prefer Not To Answer", "PMI: Skip"), NA,`Education Level: Highest Grade`),
    Education = factor(Education),
    #if this is not a numeric value, I'll need to make it one
    Alcohol = ifelse(`Alcohol: Drink Frequency Past Year` %in% c('PMI: Prefer Not To Answer', 'PMI: Skip', NA), NA, `Alcohol: Drink Frequency Past Year`),
    Smoking = ifelse(`Smoking: 100 Cigs Lifetime` %in% c('PMI: Prefer Not To Answer', 'PMI: Skip', 'PMI: Dont Know'), NA, `Smoking: 100 Cigs Lifetime`),
    Smoking = factor(Smoking)
    #I think somewhere in here I will need to process variables indicating parents AD status
  ) %>% 

  select(person_id, Income, Smoking, Race, Education, Alcohol, `Including yourself, who in your family has had dementia (includes Alzheimer's, vascular, etc.)? Select all that apply.`)

surv_covars <- surv_covars %>% 
  mutate(
    Father.Alzheimers = ifelse(grepl('Father', `Including yourself, who in your family has had dementia (includes Alzheimer's, vascular, etc.)? Select all that apply.`), 1, 0),
    Mother.Alzheimers = ifelse(grepl('Mother', `Including yourself, who in your family has had dementia (includes Alzheimer's, vascular, etc.)? Select all that apply.`), 1, 0)
  ) %>% 
  select(-`Including yourself, who in your family has had dementia (includes Alzheimer's, vascular, etc.)? Select all that apply.`)

#mean(surv_covars_1$Mother.Alzheimers)



## Add predefined snippets, which will be used to calculated current age since birth


In [ ]:
lapply(c('viridis', 'ggthemes', 'skimr'),
       function(pkg_name) { if(! pkg_name %in% installed.packages()) { install.packages(pkg_name)} } )

library(viridis)    # A nice color scheme for plots.
library(ggthemes)   # Common themes to change the look and feel of plots.
library(scales)     # Graphical scales map data to aesthetics in plots.
library(skimr)      # Better summaries of data.
library(lubridate)  # Date library from the tidyverse.
library(tidyverse)  # Data wrangling packages.
library(bigrquery)  # Data extraction from Google BigQuery

## Plot setup.
theme_set(theme_bw(base_size = 14)) # Default theme for plots.

#' Returns a data frame with a y position and a label, for use annotating ggplot boxplots.
#'
#' @param d A data frame.
#' @return A data frame with column y as max and column label as length.
get_boxplot_fun_data <- function(df) {
  return(data.frame(y = max(df), label = stringr::str_c('N = ', length(df))))
}

# Use snippet 'add_age_to_demographics' to calculate the age of people in your demographics.
# It assumes the 'Setup' snippet has been executed.
# It also assumes that you got your demographics dataframe from Dataset Builder

# Note: This snippet calculates current age and does not take into account whether the person is already dead


## -----[ CHANGE THE DATAFRAME NAME(S) `YOUR_DATASET_NAME_person_df` TO MATCH YOURS FROM DATASET BUILDER] -----
dataset_20265581_person_df <- dataset_20265581_person_df %>%
                mutate_if(is.list, as.character) %>%
                mutate(Age = year(today()) - year(dataset_20265581_person_df$date_of_birth))



## Add age and sex to dataframe


In [ ]:
age_sex_df <- dataset_20265581_person_df %>% select(person_id, Age, sex_at_birth) %>% mutate(Sex = factor(sex_at_birth))

#replace prefer not to answer with NA to prepare for imputation
age_sex_df$Sex <- as.character(age_sex_df$Sex)
age_sex_df$Sex[age_sex_df$Sex %in% c("I prefer not to answer", "PMI: Skip")] <- NA
age_sex_df$Sex <- factor(age_sex_df$Sex)



##Perform datacleaning for drugs


In [ ]:
library(dplyr)
library(stringr)

prescrip_data <- dataset_20265581_drug_df

# Define CVD drug categories by name
cvd_drugs <- list(
  statins = c('simvastatin', 'rosuvastatin', 'atorvastatin', 'pravastatin', 'fluvastatin'),
  diuretics = c("chlorothiazide", "chlorthalidone", "polythiazide", "indapamide", "metolazone", "bumetanide", "furosemide", "amiloride", "triamterene"), 
  beta_blockers = c("acebutolol", "betaxolol", "bisoprolol", "metoprolol", "nadolol", 
                   "propranolol", "timolol", "atenolol", "penbutolol", "pindolol", 
                   "carvedilol", "labetalol"),
  angiotensin_antagonists = c("candesartan", "eprosartan", "irbesartan", "losartan", "olmesartan", "telmisartan", "valsartan"),
  calcium_channel_blockers = c("diltiazem", "verapamil", "amlodipine", "felodipine", "isradipine", "nicardipine", "nisoldipine"),
  anticoagulants = c("heparin", "warfarin")
)

# Initialize indicator columns for each drug class
for(class in names(cvd_drugs)) {
  prescrip_data[[class]] <- FALSE
}

# For each class, flag rows that mention any of its drugs in 'standard_concept_name'
for(class in names(cvd_drugs)) {
  pattern <- paste0("\\b(", paste(cvd_drugs[[class]], collapse = "|"), ")\\b")
  prescrip_data[[class]] <- str_detect(tolower(prescrip_data$standard_concept_name), pattern)
}

# Optional: Keep only eid and drug class indicators
prescrip_data <- prescrip_data %>% 
  select(person_id, all_of(names(cvd_drugs))) %>% 
  group_by(person_id) %>%
  summarise(across(everything(), any, na.rm = TRUE), .groups = "drop")

#see percentages of use
lapply(prescrip_data, mean)



##Join all dataframes


In [ ]:
#this is going to need to be fixed up
clustering_data <- cvd[,1:13] %>% 
  left_join(age_sex_df) %>% 
  left_join(avg_meas_covars) %>% 
  left_join(condition_covars) %>% 
  left_join(surv_covars) %>% 
  left_join(prescrip_data) %>% 
  mutate(Diabetes = ifelse(is.na(Diabetes), FALSE, Diabetes),
         Depression.Bipolar = ifelse(is.na(Depression), FALSE, Depression)) %>% 
  select(-Depression)



##Further data cleaning


In [ ]:
#recode categorical variables that are better represented by numeric values for clustering
#income

income_map <- c("Annual Income: less 10k" = 5000, "Annual Income: 10k 25k" = 17500, "Annual Income: 25k 35k" = 30000, "Annual Income: 35k 50k" = 42500, "Annual Income: 50k 75k" = 62500, "Annual Income: 75k 100k" = 87500, "Annual Income: 100k 150k" = 125000, "Annual Income: 150k 200k" = 175000, "Annual Income: more 200k" = 200000)
clustering_data$Income <- income_map[clustering_data$Income]

#education
#this will now be represented as approximate number of years of schooling
edu_map <- c("Highest Grade: Never Attended" = 0, "Highest Grade: One Through Four" = 2.5, "Highest Grade: Five Through Eight" = 6.5, "Highest Grade: Nine Through Eleven" = 10, "Highest Grade: Twelve Or GED" = 12, "Highest Grade: College One to Three" = 14, "Highest Grade: College Graduate" = 16, "Highest Grade: Advanced Degree" = 19)
clustering_data$Education <- edu_map[clustering_data$Education]

#reclassify alcohol consumption as number of drinks per year
alcohol_map <- c("Drink Frequency Past Year: 4 or More Per Week" = 208, "Drink Frequency Past Year: 2 to 3 Per Week" = 130, "Drink Frequency Past Year: 2 to 4 Per Month" = 36, "Drink Frequency Past Year: Monthly Or Less" = 12, "Drink Frequency Past Year: Never" = 0)
clustering_data$Alcohol <- alcohol_map[clustering_data$Alcohol]

#replace missing drug info with FALSE
clustering_data[ , 28:33] <- lapply(clustering_data[ , 28:33], function(x) ifelse(is.na(x), FALSE, x))

clustering_data <- clustering_data %>% 
  select(-sex_at_birth) %>% 
  #filter to remove younger participants
  filter(Age >= 60) %>% 
  #only include participant with male or female sex at birth
  filter(Sex %in% c('Male', 'Female'))



##Prepare data for imputation


In [ ]:
#one-hot encode indicator variables
library(fastDummies)
joined_data_hot <- dummy_cols(clustering_data, 
                              remove_most_frequent_dummy = TRUE, 
                              ignore_na = TRUE) %>% 
  #remove previous categorical variables
  select(-Sex, -Smoking, -Depression.Bipolar, -Race, -person_id)

#normalize data
joined_data_scaled <- scale(joined_data_hot)

#save center and scaling factors
mu <- attr(joined_data_scaled, "scaled:center")
sigma <- attr(joined_data_scaled, "scaled:scale")

#convert scaled data matrix to dataframe
joined_data_scaled <- as.data.frame(joined_data_scaled)



## Perform missing value imputation


In [ ]:
#run KNN on continuous variables
library(VIM)
imputed_data <- kNN(joined_data_scaled)

#descale data
imputed_descaled <- sweep(imputed_data, 2, sigma, "*")
imputed_descaled <- sweep(imputed_descaled, 2, mu, "+")


#remove imputation indicators
imputed_clean <- imputed_descaled[,1:(ncol(imputed_descaled)/2)] %>% 
  cbind(person_id = clustering_data$person_id, .) %>%
  mutate(CVD = if_any(Hypertension:Cerebral.Infarction, ~ . > 0.00001))



#write to csv file
write.csv(imputed_clean, "clustering_data.csv", row.names = FALSE)

#upload from workspace to bucket
library(allofus)
aou_workspace_to_bucket("clustering_data.csv")
